# EVA-02 Base 448 — DIMER image classification tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/eva02-classification-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/eva02-classification-pipeline/blob/main/tutorials/eva02_classification_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-timm%2Feva02__base__patch14__448-ffcc4d?style=flat)](https://huggingface.co/timm/eva02_base_patch14_448.mim_in22k_ft_in22k_in1k) [![Upstream](https://img.shields.io/badge/Upstream-baaivision%2FEVA-181717?style=flat&logo=github&logoColor=white)](https://github.com/baaivision/EVA) [![arXiv](https://img.shields.io/badge/arXiv-2303.11331-b31b1b.svg)](https://arxiv.org/abs/2303.11331)

**Profile:** `E2E`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§3.6)  
**Capability:** ImageNet-1k single-label image classification (1000 classes) and in-kernel fine-tuning using the pinned EVA-02 Base patch-14 448 px weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/eva02_classification_pipeline/pipeline.py` at revision `33122bd93857`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `81063ecfe9c381a16a19d06f396d6c7011aa426a` (~348 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

At inference the pipeline **squash-resizes every input to a fixed 448 x 448** (aspect ratio is not preserved — a non-square image is stretched, not cropped), normalises with the CLIP mean/std from the snapshot config, runs one forward pass of the vision transformer, and applies a softmax over 1000 logits. **In-kernel fine-tuning:** this notebook demonstrates both zero-shot base inference on ImageNet-1k classes and 100% in-kernel classification head fine-tuning on custom classes using PyTorch AdamW optimization and cross-entropy loss. The upstream checkpoint supplies the pretrained backbone weights, and the carried pipeline module adds snapshot verification, input validation, head adaptation via `fit`, fresh-boundary reload verification, and the `top_k_accuracy`, `validate_inputs` and `evaluation_report` helpers. The default sample is a synthetic image generated in code; its prediction is demonstration (plumbing) evidence, not a production-quality or benchmark claim.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, generate a deliberately non-square synthetic input and validate it into an input manifest, run the supported task, read the argmax decision and the uncalibrated top-k softmax scores correctly, execute 100% in-kernel fine-tuning on custom classes with AdamW and cross-entropy, export and fresh-reload fine-tuned artifacts, exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` only when a ground-truth class index exists and `not-measurable` otherwise, and export machine-readable outputs plus provenance.

**This notebook does not demonstrate:** object detection, segmentation, multi-label tagging, OCR, open-vocabulary classification, feature/embedding extraction (the DINOv2 sibling covers that). The base label space is fixed to the 1000 ImageNet-1k classes; an image whose subject is outside that space still receives a label unless adapted via in-kernel fine-tuning.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. This is the heaviest of the DIMER timm classifiers (107 GMACs per 448 px image): on the model card's GPU (RTX 5070 Ti) the verified snapshot loaded in 6.4 s and one prediction took 0.8 s; **CPU works but is slow** — the card records no CPU figure and expects it to be tens of times slower than the 224 px siblings, so allow minutes, not seconds, for the single default prediction on a hosted CPU runtime. The pinned `torch==2.14.0` install and the 348 MB checkpoint are the largest downloads of the run.
- **Knowledge:** basic Python and PIL image handling; what a softmax over class logits is and why it is not a calibrated probability.
- **Data:** the default sample is a deterministic 320 x 240 RGB gradient generated in code — deliberately **not square**, so the squash to 448 x 448 is visible in the printed aspect ratio — so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image file decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, longest side at most 4096 px; it is squash-resized to 448 x 448 regardless of its aspect ratio. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `timm/eva02_base_patch14_448.mim_in22k_ft_in22k_in1k` snapshot (~348 MB in total) at revision `81063ecfe9c3…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `timm` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'timm==1.0.29',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'eva02-classification-pipeline',
    'repository_revision': '33122bd938571421562916144d40f3004ccf3abd',
    'embedded_module': 'src/eva02_classification_pipeline/pipeline.py',
    'embedded_modules': ['src/eva02_classification_pipeline/pipeline.py'],
    'module_sha256': 'd43692f013a18538e6fad8f6f4240c941b93056011480c52703d9d5b9085407b',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, timm
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'timm': timm.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/eva02_classification_pipeline/` @ `33122bd93857`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/eva02_classification_pipeline/pipeline.py`

In [ ]:
"""ImageNet-1k classification with the pinned EVA-02 Base 448 px checkpoint named by ``MODEL_ID``.

EVA-02 Base, patch 14, fixed 448x448 input (squash resize, CLIP mean/std), mean-pooled ViT, 1000-way head.
The class loads weights only from a digest-verified local snapshot (``weights/<key>/``) or,
when explicitly allowed, from the Hugging Face Hub at the pinned revision. Preprocessing is
the upstream ``pretrained_cfg`` (resize/crop/normalize) resolved through ``timm.data``.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "timm/eva02_base_patch14_448.mim_in22k_ft_in22k_in1k"
MODEL_REVISION = "81063ecfe9c381a16a19d06f396d6c7011aa426a"
MODEL_LICENSE = "mit"
MODEL_KEY = "eva02-base-448"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHTS_FILE = "model.safetensors"
CONFIG_FILE = "config.json"

NUM_CLASSES = 1000
MAX_IMAGE_SIDE = 4096  # pixels; larger images are rejected before any decode-to-tensor work
MAX_BATCH = 64  # images per predict() call
DEFAULT_TOP_K = 5
DECISION_RULE = "argmax"  # the label reported as `predicted_index` is the softmax argmax; no threshold


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _hub_reference(model_id: str, revision: str) -> str:
    """timm's ``hf-hub:owner/name@revision`` form; ``hf_split`` passes ``revision=`` to hf_hub_download."""
    return f"hf-hub:{model_id}@{revision}"


def top_k_accuracy(predictions: Sequence[Any], targets: Sequence[int], k: int = 1) -> float:
    """Fraction of items whose target index is among the first ``k`` predicted indices.

    ``predictions`` may be the per-image dicts returned by ``predict`` or plain index sequences.
    """
    if len(predictions) != len(targets):
        raise ValueError("predictions and targets must have the same length")
    if not predictions:
        raise ValueError("predictions must not be empty")
    if not isinstance(k, int) or k < 1:
        raise ValueError("k must be a positive integer")
    hits = 0
    for pred, target in zip(predictions, targets, strict=True):
        ranked = pred["top_k"] if isinstance(pred, Mapping) else pred
        indices = [int(item["index"]) if isinstance(item, Mapping) else int(item) for item in ranked]
        hits += int(target in indices[:k])
    return hits / len(predictions)


INPUT_SCHEMA: dict[str, Any] = {
    "input": "PIL.Image.Image or a sequence of them; any mode, converted to RGB",
    "image_side_px": [1, MAX_IMAGE_SIDE],
    "batch": [1, MAX_BATCH],
    "top_k": [1, NUM_CLASSES],
    "preprocessing": "convert to RGB, squash-resize to 448x448 (crop_mode squash, bicubic), CLIP mean/std",
}


def _check_inputs(images: Any, top_k: int) -> list[Image.Image]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the images as a list."""
    if isinstance(images, Image.Image):
        images = [images]
    if not isinstance(images, Sequence) or isinstance(images, str | bytes):
        raise TypeError("images must be a PIL.Image.Image or a sequence of them")
    if not 1 <= len(images) <= MAX_BATCH:
        raise ValueError(f"batch size must be between 1 and MAX_BATCH={MAX_BATCH}, got {len(images)}")
    for image in images:
        if not isinstance(image, Image.Image):
            raise TypeError(f"each image must be a PIL.Image.Image, got {type(image).__name__}")
        width, height = image.size
        if width < 1 or height < 1 or max(width, height) > MAX_IMAGE_SIDE:
            raise ValueError(f"image side outside 1..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: {image.size}")
    if isinstance(top_k, bool) or not isinstance(top_k, int):
        raise TypeError("top_k must be an int")
    if not 1 <= top_k <= NUM_CLASSES:
        raise ValueError(f"top_k must be between 1 and NUM_CLASSES={NUM_CLASSES}")
    return list(images)


def validate_inputs(
    images: Image.Image | Sequence[Image.Image],
    top_k: int = DEFAULT_TOP_K,
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Rejection is reported by raising exactly as ``predict`` would; a caller that wants the
    finding recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    checked = _check_inputs(images, top_k)
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per image")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[i] if names else f"image-{i}", "mode": image.mode, "size": list(image.size)}
            for i, image in enumerate(checked)
        ],
        "top_k": top_k,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], targets: Sequence[int] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``targets`` (one ImageNet-1k index per prediction) the report carries ``top_k_accuracy``
    at k=1 and k=5 as sample-sanity evidence; without them the verdict is ``not-measurable`` and
    the report says what labelled data would make the task measurable.
    """
    predictions = result["predictions"]
    base = {
        "task": "imagenet-1k single-label classification",
        "decision_rule": result.get("decision_rule", DECISION_RULE),
        "sample_kind": sample_kind,
        "n_predictions": len(predictions),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if targets is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth class index was supplied for the evaluated images",
            "needs": (
                "labelled photographs with ImageNet-1k class indices (0-999), e.g. a held-out sample of your "
                "own data, scored with top_k_accuracy against the majority-class baseline of that sample"
            ),
        }
    top_k = int(result.get("top_k", DEFAULT_TOP_K))
    ks = sorted({1, min(5, top_k)})
    return {
        **base,
        "metrics": [
            {
                "id": "top_k_accuracy",
                "k": k,
                "value": top_k_accuracy(predictions, list(targets), k=k),
                "estimation": "single sample, no dispersion estimate",
            }
            for k in ks
        ],
        "verdict": "sample-sanity",
        "reason": f"{len(predictions)} labelled image(s) from the tutorial sample; not a benchmark",
        "needs": "a labelled evaluation set from the deployment domain for any generalisable accuracy claim",
    }


@dataclass
class EVA02ClassificationPipeline:
    """``_runner`` maps a float tensor (N, 3, H, W) to logits (N, NUM_CLASSES); injectable for tests."""

    _runner: Callable[[Any], Any]
    _transform: Callable[[Image.Image], Any]
    device: str = "cpu"
    labels: tuple[str, ...] = ()
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> EVA02ClassificationPipeline:
        import timm
        import torch
        from timm.data import ImageNetInfo, create_transform, resolve_model_data_config

        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        arch_name = MODEL_ID.split("/", 1)[1]
        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")

        if (root / "model-config.json").is_file() and (root / WEIGHTS_FILE).is_file():
            from safetensors.torch import load_file
            with open(root / "model-config.json", encoding="utf-8") as fh:
                cfg = json.load(fh)
            num_classes = cfg.get("num_classes", len(cfg.get("class_names", [])))
            labels = tuple(cfg.get("class_names", [f"class_{i}" for i in range(num_classes)]))
            model = timm.create_model(arch_name, pretrained=False, num_classes=num_classes)
            model.load_state_dict(load_file(root / WEIGHTS_FILE, device=str(resolved_device)), strict=True)
            source = "fine-tuned-artifact"
            data_config = cfg.get("data_config") or resolve_model_data_config(model)
            transform = create_transform(**data_config, is_training=False)
        elif (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            with open(root / CONFIG_FILE, encoding="utf-8") as fh:
                config = json.load(fh)
            snapshot_name = f"{config['architecture']}.{config['pretrained_cfg']['tag']}"
            if snapshot_name != arch_name:
                raise ValueError(f"snapshot config names {snapshot_name!r}, expected {arch_name!r}")
            overlay = dict(config["pretrained_cfg"])
            overlay["file"] = str(root / WEIGHTS_FILE)  # 'file' takes precedence over hf_hub_id in timm
            model = timm.create_model(
                arch_name, pretrained=True, pretrained_cfg_overlay=overlay, num_classes=NUM_CLASSES
            )
            source = "local-snapshot"
            data_config = resolve_model_data_config(model)
            transform = create_transform(**data_config, is_training=False)
            info = ImageNetInfo(subset="imagenet-1k")
            labels = tuple(info.index_to_description(i) for i in range(info.num_classes()))
        elif allow_download:
            model = timm.create_model(
                _hub_reference(MODEL_ID, revision=MODEL_REVISION), pretrained=True, num_classes=NUM_CLASSES
            )
            source = "hf-hub"
            data_config = resolve_model_data_config(model)
            transform = create_transform(**data_config, is_training=False)
            info = ImageNetInfo(subset="imagenet-1k")
            labels = tuple(info.index_to_description(i) for i in range(info.num_classes()))
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        model = model.eval().to(resolved_device)

        def runner(batch: Any) -> Any:
            with torch.inference_mode():
                return model(batch.to(resolved_device))

        return cls(runner, transform, resolved_device, labels, source)

    @classmethod
    def fit(
        cls,
        train_images: Sequence[Image.Image],
        train_targets: Sequence[int],
        val_images: Sequence[Image.Image],
        val_targets: Sequence[int],
        class_names: Sequence[str],
        *,
        epochs: int = 1,
        batch_size: int = 4,
        learning_rate: float = 1e-4,
        weight_decay: float = 0.01,
        seed: int = 20260910,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        output_dir: str | Path | None = None,
    ) -> tuple[EVA02ClassificationPipeline, dict[str, Any]]:
        """Fine-tune the EVA-02 model on custom classes 100% in-kernel."""
        import timm
        import torch
        import torch.nn.functional as F
        from safetensors.torch import load_file, save_file
        from torch.utils.data import DataLoader, Dataset

        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        num_classes = len(class_names)
        arch_name = MODEL_ID.split("/", 1)[1]

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        model = timm.create_model(arch_name, pretrained=False, num_classes=num_classes)
        weights_path = root / WEIGHTS_FILE
        if weights_path.is_file():
            sd = load_file(weights_path)
            backbone_sd = {k: v for k, v in sd.items() if not k.startswith("head.")}
            model.load_state_dict(backbone_sd, strict=False)

        model.to(resolved_device)
        data_config = timm.data.resolve_model_data_config(model)
        train_transform = timm.data.create_transform(**data_config, is_training=True)
        eval_transform = timm.data.create_transform(**data_config, is_training=False)

        class ImageDataset(Dataset):
            def __init__(self, imgs: Sequence[Image.Image], targets: Sequence[int], transform_fn: Any):
                self.imgs = list(imgs)
                self.targets = list(targets)
                self.transform_fn = transform_fn

            def __len__(self) -> int:
                return len(self.imgs)

            def __getitem__(self, idx: int) -> tuple[Any, int]:
                img = self.imgs[idx].convert("RGB")
                tensor = self.transform_fn(img)
                return tensor, self.targets[idx]

        train_loader = DataLoader(
            ImageDataset(train_images, train_targets, train_transform),
            batch_size=batch_size,
            shuffle=True,
        )
        val_loader = DataLoader(
            ImageDataset(val_images, val_targets, eval_transform),
            batch_size=batch_size,
            shuffle=False,
        )

        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        history = []

        for epoch in range(epochs):
            model.train()
            train_loss_sum, train_count = 0.0, 0
            for inputs, targets in train_loader:
                inputs = inputs.to(resolved_device)
                targets = targets.to(resolved_device)
                optimizer.zero_grad(set_to_none=True)
                outputs = model(inputs)
                loss = F.cross_entropy(outputs, targets)
                loss.backward()
                optimizer.step()
                train_loss_sum += float(loss.detach().cpu()) * targets.numel()
                train_count += targets.numel()

            model.eval()
            val_loss_sum, val_correct, val_count = 0.0, 0, 0
            with torch.no_grad():
                for inputs, targets in val_loader:
                    inputs = inputs.to(resolved_device)
                    targets = targets.to(resolved_device)
                    outputs = model(inputs)
                    loss = F.cross_entropy(outputs, targets)
                    val_loss_sum += float(loss.detach().cpu()) * targets.numel()
                    val_correct += int((outputs.argmax(dim=-1) == targets).sum())
                    val_count += targets.numel()

            history.append({
                "epoch": epoch + 1,
                "train_loss": train_loss_sum / max(1, train_count),
                "val_loss": val_loss_sum / max(1, val_count),
                "val_accuracy": val_correct / max(1, val_count),
            })

        if output_dir is not None:
            out_path = Path(output_dir)
            out_path.mkdir(parents=True, exist_ok=True)
            safetensors_path = out_path / WEIGHTS_FILE
            save_file({k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()}, safetensors_path)
            config_payload = {
                "architecture": "eva02_base_patch14_448",
                "num_classes": num_classes,
                "class_names": list(class_names),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
                "data_config": data_config,
            }
            with open(out_path / "model-config.json", "w", encoding="utf-8") as fh:
                json.dump(config_payload, fh, indent=2)

        def runner(batch: Any) -> Any:
            with torch.inference_mode():
                return model(batch.to(resolved_device))

        pipeline = cls(runner, eval_transform, resolved_device, tuple(class_names), "fine-tuned")
        return pipeline, {"history": history, "class_names": list(class_names), "device": resolved_device}

    def _validate(self, images: Any, top_k: int) -> list[Image.Image]:
        return _check_inputs(images, top_k)

    def predict(
        self, images: Image.Image | Sequence[Image.Image], top_k: int = DEFAULT_TOP_K
    ) -> dict[str, Any]:
        """Classify images; ``score`` is a softmax score, not a calibrated probability."""
        import torch

        batch_images = self._validate(images, top_k)
        batch = torch.stack([self._transform(image.convert("RGB")) for image in batch_images])
        logits = self._runner(batch)
        expected_classes = len(self.labels) if self.labels else NUM_CLASSES
        if not isinstance(logits, torch.Tensor) or logits.shape != (len(batch_images), expected_classes):
            raise RuntimeError(f"runner must return a tensor of shape (batch, {expected_classes})")
        scores = torch.softmax(logits.float(), dim=-1).cpu()
        values, indices = torch.topk(scores, k=top_k, dim=-1)
        predictions = []
        for image_values, image_indices in zip(values.tolist(), indices.tolist(), strict=True):
            image_values = [float(s) for s in image_values]
            ranked = [
                {"label": self.labels[i] if i < len(self.labels) else str(i), "index": i, "score": s}
                for s, i in zip(image_values, image_indices, strict=True)
            ]
            best = ranked[0]
            predictions.append(
                {"predicted_index": best["index"], "predicted_label": best["label"], "top_k": ranked}
            )
        return {
            "predictions": predictions,
            "top_k": top_k,
            "decision_rule": DECISION_RULE,
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `3`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `81063ecfe9c3…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `EVA02ClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "eva02-base-448",
  "modelId": "timm/eva02_base_patch14_448.mim_in22k_ft_in22k_in1k",
  "revision": "81063ecfe9c381a16a19d06f396d6c7011aa426a",
  "files": [
    {
      "path": "README.md",
      "bytes": 5454,
      "sha256": "b347298844860ff288638f1d7c0baece71d9fe1cd796eefbfbf3f0780175ba07"
    },
    {
      "path": "config.json",
      "bytes": 653,
      "sha256": "d62e73f578eb363032000b6bfcc57764c6a5d10ee2562cbfe3f105d795e1c4be"
    },
    {
      "path": "model.safetensors",
      "bytes": 348492484,
      "sha256": "533937d6f9f8f8d4f50627ef0d00829a5015861a5b67e1c69c5a5e45b7dc2609"
    }
  ],
  "totalBytes": 348498591
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = EVA02ClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Generate the synthetic sample or optional BYOD

The default sample is **synthetic**: a deterministic 320 x 240 RGB gradient built in code (red ramps left to right, green top to bottom, blue is their mean), so it needs no download and its SHA-256 is printed for the record. It is deliberately **not square** so that the squash to 448 x 448 is visible in the printed aspect ratio. A gradient is not a photograph of any ImageNet class, so it has **no ground truth**: whatever label the model returns is a sanity check that the input contract, preprocessing and forward pass work, not a correctness measurement. BYOD is optional and disabled by default; when enabled, upload one image file and, if you know its ImageNet-1k class index (0–999), set `GROUND_TRUTH_INDEX` so the evaluation step can compute `top_k_accuracy`. Leave it at `-1` when the label is unknown. Look for a dictionary naming the sample kind, its size, aspect ratio and digest, and whether a ground-truth index was supplied.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image

USE_BYOD = False  # @param {type:"boolean"}
GROUND_TRUTH_INDEX = -1  # @param {type:"integer"}
SAMPLE_WIDTH = 320
SAMPLE_HEIGHT = 240

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic gradient, deliberately non-square so the squash to 448x448 is visible.
    red = np.tile(np.linspace(0.0, 255.0, SAMPLE_WIDTH), (SAMPLE_HEIGHT, 1))
    green = np.tile(np.linspace(0.0, 255.0, SAMPLE_HEIGHT), (SAMPLE_WIDTH, 1)).T
    blue = (red + green) / 2.0
    array = np.rint(np.stack([red, green, blue], axis=-1)).astype(np.uint8)
    image = Image.fromarray(array, mode='RGB')
    image_name = f'synthetic_gradient_{SAMPLE_WIDTH}x{SAMPLE_HEIGHT}.png'
    sample_kind = 'synthetic'

if GROUND_TRUTH_INDEX != -1 and not 0 <= GROUND_TRUTH_INDEX < NUM_CLASSES:
    raise ValueError(f'GROUND_TRUTH_INDEX must be -1 (unknown) or an ImageNet-1k class index in 0..{NUM_CLASSES - 1}, got {GROUND_TRUTH_INDEX}.')
ground_truth = None if GROUND_TRUTH_INDEX == -1 else GROUND_TRUTH_INDEX
sample_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
aspect_ratio = round(image.size[0] / image.size[1], 3)
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'aspect_ratio': aspect_ratio, 'rgb_sha256': sample_sha256, 'ground_truth_index': ground_truth})

## 5. Validate the input → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `predict` applies — type, batch size 1..`MAX_BATCH`, image side 1..`MAX_IMAGE_SIDE` px, `top_k` 1..`NUM_CLASSES` — and returns an **input manifest** naming the schema and ceilings, each input's observed mode and size, and the verdict. The manifest is written to `outputs/eva02_classification_input_manifest.json`. To show what rejection looks like, the cell also validates a deliberately oversized image and records the pipeline's own error message as a finding. **What the pipeline changes about your image:** it converts to RGB and squash-resizes to exactly 448 x 448 (`crop_mode: "squash"`, `crop_pct: 1.0` in the snapshot config) — nothing is cropped or dropped, but a non-square image is distorted in proportion to the aspect ratio printed above. The notebook itself does not resize, crop, or subsample.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'NUM_CLASSES': NUM_CLASSES, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_BATCH': MAX_BATCH}})
input_manifest = validate_inputs(image, top_k=5, names=[image_name])
# Demonstrate rejection on an input that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(Image.new('RGB', (MAX_IMAGE_SIDE + 1, 8)))
except ValueError as exc:
    input_manifest['findings'].append({'input': 'oversized-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/eva02_classification_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Classify

`predict` returns, per image, `predicted_index`/`predicted_label` and a `top_k` list of `{label, index, score}` entries **ordered by descending score** — rank position is the class ordering, and the exported files preserve it. The decision rule is `argmax` over the 1000 softmax scores (`decision_rule` in the result); the pipeline ships no acceptance threshold, and `score` is a softmax over uncalibrated logits, **not a calibrated probability**. A deployment that needs an abstain option must choose its own score cut-off on its own labelled data — downstream calibration is the caller's responsibility. Inference is deterministic given the same weights, device and library versions (no sampling, `model.eval()`); CPU and CUDA kernel choices can reorder near-tied classes. Look for the ranked top-5 list; on the gradient expect a low top-1 score spread across unrelated classes (the model card's smoke run labelled one `screen, CRT screen` at score 0.013). On a CPU runtime this cell is the slow one — minutes, not seconds.

In [ ]:
result = pipe.predict(image, top_k=5)
prediction = result['predictions'][0]
print({'decision_rule': result['decision_rule'], 'predicted_index': prediction['predicted_index'], 'predicted_label': prediction['predicted_label'], 'device': result['device'], 'source': result['source']})
for rank, item in enumerate(prediction['top_k'], start=1):
    print(f"{rank:>2}. index {item['index']:>4}  score {item['score']:.4f}  {item['label']}")

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. When a ground-truth class index was supplied in Section 4 it carries `top_k_accuracy` (the repository's metric helper) at k=1 and k=5 with the verdict `sample-sanity` — a single-image tutorial figure (0.0 or 1.0) with no dispersion estimate, and nothing that can be generalised to a domain. On the synthetic default sample no metric exists, so the verdict is `not-measurable` and the report states what would make the task measurable: labelled photographs with ImageNet-1k class indices, for example a held-out sample of your own data scored against its majority-class baseline. The upstream 88.692 % top-1 / 98.722 % top-5 at 448 px on the ImageNet-1k validation set is an upstream claim quoted by the model card, not something this notebook measures. The report is written to `outputs/eva02_classification_evaluation_report.json`.

In [ ]:
targets = None if ground_truth is None else [ground_truth]
report = evaluation_report(result, targets, sample_kind=sample_kind)
with open('outputs/eva02_classification_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No ground-truth class index was supplied, so top_k_accuracy is not computed; the prediction above is sanity evidence only.')

## 8. Dataset acquisition and in-kernel fine-tuning

`pipe.fit(...)` implements 100% in-kernel head adaptation: it replaces the 1000-class head with a new linear classifier sized to the target classes (`len(class_names)`), initializes from the verified backbone weights without shape collision, and trains with `torch.optim.AdamW` and cross-entropy loss. No external worker repositories, CLI subprocesses, or unpinned dependencies are invoked.

**Data Acquisition & BYOD:**
- **Default Sample Dataset (`USE_BYOD_DATASET = False`):** Automatically downloads and verifies the SHA-256 digest of [`Cleanlab/cifar-10-subset`](https://huggingface.co/datasets/Cleanlab/cifar-10-subset) (MIT license, ~986 KB, 400 images across 2 balanced classes: `frog` and `truck`). A balanced subset is loaded for rapid tutorial smoke, and automatically split into `train` (80%) and `val` (20%). If the runtime is air-gapped, it gracefully falls back to deterministic synthetic stripes.
- **Bring Your Own Data (`USE_BYOD_DATASET = True`):** Upload a `.zip` archive containing either explicit `train/` and `val/` directories or un-split class folders (in which case a seeded 80/20 stratified split is performed automatically). Enforces >= 2 classes and >= 2 images per class.

Deployable fine-tuned artifacts (`model.safetensors` and `model-config.json`) are written atomically to `outputs/eva02_classification_finetuned`.

In [ ]:
import hashlib
import io
import random
import urllib.request
import zipfile

USE_BYOD_DATASET = False  # @param {type:"boolean"}
SAMPLE_DATASET_URL = 'https://huggingface.co/datasets/Cleanlab/cifar-10-subset/resolve/bb5a7aabf1d14d2d1e3e49d0d8f917bda3622f75/CIFAR-10-subset.zip'
SAMPLE_DATASET_SHA256 = '66f90a4f87d865e8eb653b62f10e754684075a32314177de76832349d4b1fb19'
SEED = 42
VALIDATION_SPLIT = 0.2
SUBSET_PER_CLASS = 16  # balanced sample per class for fast in-kernel smoke

train_images, train_targets = [], []
val_images, val_targets = [], []
zip_bytes = None

if USE_BYOD_DATASET:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one dataset .zip archive.')
    zip_name, zip_bytes = next(iter(uploaded.items()))
    if not zip_name.lower().endswith('.zip'):
        raise ValueError(f'BYOD archive must be a .zip file, got {zip_name}')
    dataset_source = f'user upload: {zip_name}'
else:
    try:
        req = urllib.request.Request(SAMPLE_DATASET_URL, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=30) as resp:
            zip_bytes = resp.read()
        actual_sha = hashlib.sha256(zip_bytes).hexdigest()
        if actual_sha != SAMPLE_DATASET_SHA256:
            raise ValueError(f'Sample dataset SHA-256 mismatch: expected {SAMPLE_DATASET_SHA256}, got {actual_sha}')
        dataset_source = f'Cleanlab/cifar-10-subset (MIT license, sha256:{actual_sha[:16]}...)'
    except Exception as exc:
        print(f'Warning: public sample dataset download failed ({exc}); falling back to deterministic synthetic dataset.')
        zip_bytes = None
        dataset_source = 'synthetic stripes fallback'

if zip_bytes is not None:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        # Security audit: reject directory traversal and absolute paths
        for info in z.infolist():
            if '..' in info.filename or info.filename.startswith(('/', '\\')):
                raise ValueError(f'Security violation: illegal path in zip archive: {info.filename}')
        names = [n for n in z.namelist() if n.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp')) and not n.startswith('__MACOSX')]
        if not names:
            raise ValueError('No supported image files (.png, .jpg, .jpeg, .webp, .bmp) found in archive.')

        has_train = any('train/' in n.lower() for n in names)
        has_val = any('val/' in n.lower() or 'valid/' in n.lower() for n in names)

        def _class_from_path(p):
            parts = p.strip('/').split('/')
            return parts[-2] if len(parts) >= 2 else 'unknown'

        if has_train and has_val:
            train_names = [n for n in names if 'train/' in n.lower()]
            val_names = [n for n in names if 'val/' in n.lower() or 'valid/' in n.lower()]
            CUSTOM_CLASSES = sorted(list({_class_from_path(n) for n in train_names}))
            if len(CUSTOM_CLASSES) < 2:
                raise ValueError(f'Classification requires at least 2 distinct classes, found {CUSTOM_CLASSES}')
            cls_map = {c: i for i, c in enumerate(CUSTOM_CLASSES)}
            for n in train_names:
                cls = _class_from_path(n)
                if cls in cls_map:
                    train_images.append(Image.open(io.BytesIO(z.read(n))).convert('RGB'))
                    train_targets.append(cls_map[cls])
            for n in val_names:
                cls = _class_from_path(n)
                if cls in cls_map:
                    val_images.append(Image.open(io.BytesIO(z.read(n))).convert('RGB'))
                    val_targets.append(cls_map[cls])
        else:
            class_to_files = {}
            for n in names:
                cls = _class_from_path(n)
                class_to_files.setdefault(cls, []).append(n)
            CUSTOM_CLASSES = sorted(list(class_to_files.keys()))
            if len(CUSTOM_CLASSES) < 2:
                raise ValueError(f'Classification requires at least 2 distinct classes to train, found {len(CUSTOM_CLASSES)}: {CUSTOM_CLASSES}')
            cls_map = {c: i for i, c in enumerate(CUSTOM_CLASSES)}
            rng = random.Random(SEED)
            for cls, files in class_to_files.items():
                if len(files) < 2:
                    raise ValueError(f'Class {cls!r} has fewer than 2 images ({len(files)}); cannot perform train/val split.')
                f_list = list(files)
                rng.shuffle(f_list)
                if not USE_BYOD_DATASET and SUBSET_PER_CLASS:
                    f_list = f_list[:SUBSET_PER_CLASS]
                n_val = max(1, int(len(f_list) * VALIDATION_SPLIT))
                val_files = f_list[:n_val]
                train_files = f_list[n_val:]
                for f in train_files:
                    train_images.append(Image.open(io.BytesIO(z.read(f))).convert('RGB'))
                    train_targets.append(cls_map[cls])
                for f in val_files:
                    val_images.append(Image.open(io.BytesIO(z.read(f))).convert('RGB'))
                    val_targets.append(cls_map[cls])
else:
    CUSTOM_CLASSES = ['synthetic_horizontal_stripe', 'synthetic_vertical_stripe']
    for cls_idx, pattern in enumerate(['horizontal', 'vertical']):
        for i in range(6):
            arr = np.zeros((SAMPLE_HEIGHT, SAMPLE_WIDTH, 3), dtype=np.uint8)
            if pattern == 'horizontal':
                arr[::32, :, 0] = 255
                arr[:, :, 2] = (i * 30) % 255
            else:
                arr[:, ::32, 1] = 255
                arr[:, :, 2] = (i * 30) % 255
            img = Image.fromarray(arr, mode='RGB')
            if i < 4:
                train_images.append(img)
                train_targets.append(cls_idx)
            else:
                val_images.append(img)
                val_targets.append(cls_idx)

ft_output_dir = 'outputs/eva02_classification_finetuned'
fine_tuned_pipe, train_meta = pipe.fit(
    train_images=train_images,
    train_targets=train_targets,
    val_images=val_images,
    val_targets=val_targets,
    class_names=CUSTOM_CLASSES,
    epochs=1,
    batch_size=2,
    learning_rate=1e-4,
    weights_dir=WEIGHTS_DIR,
    output_dir=ft_output_dir,
)
print({
    'fine_tuning': 'complete',
    'dataset_source': dataset_source,
    'classes': CUSTOM_CLASSES,
    'train_samples': len(train_images),
    'val_samples': len(val_images),
    'epochs': len(train_meta['history']),
    'history': train_meta['history'],
})

## 9. Fresh-boundary reload and verification

To verify artifact integrity across an isolation boundary (simulating a fresh deployment or downstream container), `EVA02ClassificationPipeline.from_pretrained` loads the newly generated `model.safetensors` and `model-config.json` directly from `outputs/eva02_classification_finetuned`. The pipeline re-instantiates the architecture, restores weights with `strict=True`, configures the custom class labels, and runs inference on a held-out validation sample.

In [ ]:
reloaded_pipe = EVA02ClassificationPipeline.from_pretrained(weights_dir=ft_output_dir)
reloaded_result = reloaded_pipe.predict(val_images[0], top_k=min(2, len(CUSTOM_CLASSES)))
reloaded_pred = reloaded_result['predictions'][0]
print({
    'source': reloaded_result['source'],
    'predicted_label': reloaded_pred['predicted_label'],
    'predicted_index': reloaded_pred['predicted_index'],
    'top_k': reloaded_pred['top_k'],
})

## 10. Export outputs and provenance

Machine-readable JSON preserves the full prediction (argmax decision and the rank-ordered top-k scores), the evaluation report, the fine-tuning training history, the sample identity, aspect ratio and digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `timm`, device). The rank-ordered top-k table is also written as CSV with explicit `rank`, `index`, `label` and `score` columns so class ordering survives downstream use. Deployable fine-tuned model artifacts are published in `outputs/eva02_classification_finetuned/`. No credentials are recorded.

In [ ]:
import csv

payload = {
    'prediction': result,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'fine_tuning': {'history': train_meta['history'], 'classes': CUSTOM_CLASSES, 'reloaded_prediction': reloaded_result},
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'aspect_ratio': aspect_ratio, 'rgb_sha256': sample_sha256, 'ground_truth_index': ground_truth},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'timm': timm.__version__,
        'device': pipe.device,
    },
}
with open('outputs/eva02_classification_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/eva02_classification_top_k.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'rank', 'index', 'label', 'score'])
    for rank, item in enumerate(prediction['top_k'], start=1):
        writer.writerow([image_name, rank, item['index'], item['label'], f"{item['score']:.6f}"])
print({'outputs': sorted(os.listdir('outputs')), 'finetuned': sorted(os.listdir(ft_output_dir))})
print(['outputs/eva02_classification_finetuned/model.safetensors', 'outputs/eva02_classification_finetuned/model-config.json'])

## Interpretation and limits

The predicted label is the argmax of a softmax over the fixed 1000-class ImageNet-1k label space (or custom class names when fine-tuned); the `score` values are uncalibrated softmax outputs, not probabilities of correctness, and the pipeline ships no threshold. On the synthetic gradient the label is meaningless by construction and the evaluation report says `not-measurable`; a `top_k_accuracy` value shown for a single BYOD image is 0 or 1 and says nothing about the error rate on a domain. Every input is squash-resized to 448 x 448, so strongly non-square subjects are distorted before classification; the pipeline does not detect out-of-distribution inputs (drawings, scans, satellite tiles), blur, or capture-device drift, and it exposes no features, no detection, and no labels beyond the fixed 1000.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated input against the enforced ceilings, execute the public pipeline path, perform in-kernel fine-tuning, reload the verified artifact bundle, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, reproduction of the upstream accuracy, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. A `sha256`/`size` `ValueError` or `FileNotFoundError: snapshot file missing` in Section 3: a staged file is incomplete or altered — delete it from the working-directory `weights/eva02-base-448/` and rerun Section 3. A `ValueError` naming `MAX_IMAGE_SIDE` or `GROUND_TRUTH_INDEX`: fix the form values in Section 4 and rerun from there. A very slow Section 6 or 8 on a CPU runtime is expected for this 107-GMAC model; switch to a CUDA runtime or use the MobileNetV4 sibling for CPU latency.

**Next experiments:** enable `USE_BYOD` with a photograph of a known ImageNet class and its index to see the report switch to `sample-sanity` with `top_k_accuracy` at k=1 and k=5; supply your own multi-class dataset folder to `pipe.fit` to adapt to your domain; upload a strongly non-square version of the same photograph to observe the effect of the squash; compare the CUDA and CPU top-5 orderings on the same image to observe kernel-level variability.

## References

- Repository README: https://github.com/kurtvalcorza/eva02-classification-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/eva02-classification-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/eva02-classification-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/timm/eva02_base_patch14_448.mim_in22k_ft_in22k_in1k
- Upstream code: https://github.com/baaivision/EVA
- EVA-02 paper: https://arxiv.org/abs/2303.11331
- timm documentation: https://huggingface.co/docs/timm